# Урок 4.1. Почему RAG ломается в продакшене

Модуль 3 закончился рабочим сервисом: он ищет, отвечает, отказывается,
считает метрики и переживает падение Ollama. На демо он безупречен.
А потом его выкатывают на реальную базу знаний — и начинается.

> «RAG в демо и RAG в проде — это две разные системы с одинаковым кодом.»

В этом уроке мы **специально ломаем** наш сервис пятью типовыми способами
и учимся **видеть** поломку — потому что главная беда продового RAG не в том,
что он падает с трассировкой стека, а в том, что он **уверенно отвечает
неправильно**, и никто этого не замечает неделями.

## 🎯 Цели урока
- Разобрать пять антипаттернов, из-за которых RAG деградирует в проде:
  устаревшие данные, плохой чанкинг, нерелевантный поиск, длинный контекст,
  тихие сбои без наблюдаемости.
- Научиться **диагностировать** каждый — на живой базе «Векторики».
- Понять, какие уроки модуля 4 лечат какую болезнь (карта впереди).


## Где мы в курсе

Модуль 3 дал **механику**: как собрать работающий RAG. Модуль 4 — про то,
как сделать его **надёжным, безопасным и наблюдаемым**. Карта модуля:

| Урок | Болезнь, которую лечит | Лекарство |
|---|---|---|
| **4.1 (сейчас)** | RAG тихо деградирует | диагностика антипаттернов |
| 4.2 | документ атакует модель | санитизация, контроль доступа, PII |
| 4.3 | сбой видят через неделю | логи, Prometheus-метрики, трейсинг |
| 4.4 | поиск промахивается | reranking, RRF, multi-query |
| 4.5 | все видят чужие документы | метаданные, права, мультитенантность |
| 4.6 | закрепление | квиз + практика |

Сквозной пример прежний — база знаний вымышленной компании **«Векторика»**.
Но корпус подрос: теперь в нём есть **две версии** документа об отпуске,
**конфиденциальные** зарплатные вилки, **постмортем с ПДн** и специально
**отравленный** community-документ. На них и будем ломаться.


## Анатомия поломки: где рвётся конвейер

Вспомним конвейер RAG из модуля 3 и отметим, где именно он ломается в проде:

```
                        ┌─────────────── АНТИПАТТЕРНЫ ───────────────┐
  документы ──▶ чанкинг ──▶ индекс ──▶ поиск ──▶ промпт ──▶ LLM ──▶ ответ
      │           │                      │          │
      ▼           ▼                      ▼          ▼
  (1) устарели  (2) режем       (3) находим не то  (4) контекст
     — версий     по-живому        — семантика/       раздут —
     нет          — смысл рвётся    лексика мимо       «lost in
                                                        the middle»

  (5) И ГЛАВНОЕ: всё это происходит ТИХО. Метрик нет — поломки не видно.
```

Заметьте: ни один из пяти не бросает исключение. Сервис жив, `/healthz`
зелёный, латентность в норме. Просто ответы стали хуже. Поехали по порядку.


In [ ]:
import sys
import os
import json
from pathlib import Path

# Путь к пакету rag_service (запускайте ноутбук из корня модуля 04-advanced-rag)
MODULE_DIR = Path.cwd()
if not (MODULE_DIR / "rag_service").exists() and (MODULE_DIR.parent / "rag_service").exists():
    MODULE_DIR = MODULE_DIR.parent
sys.path.insert(0, str(MODULE_DIR))
os.environ.setdefault("RAG_JSON_LOGS", "0")  # человекочитаемые логи в ноутбуке

In [ ]:
from rag_service.config import settings, DATA_DIR
from rag_service.store import KnowledgeBase
from rag_service.llm import OllamaLLM

# Флаги окружения: ячейки с Qdrant/LLM ниже проверяют их и мягко пропускаются,
# если инфраструктура не поднята (у вас без Docker/Ollama урок всё равно читается).
kb = KnowledgeBase()
QDRANT_UP = kb.available() and kb.collection_exists()
OLLAMA_UP = OllamaLLM().available()
print(f"Qdrant: {'✓' if QDRANT_UP else '✗ (docker compose up -d + python -m rag_service.ingest)'}")
print(f"Ollama: {'✓' if OLLAMA_UP else '✗ (ollama serve + ollama pull qwen2.5:3b)'}")
print(f"коллекция: {settings.collection}, чанков: {kb.count() if QDRANT_UP else '—'}")

> **Если Qdrant пуст.** Один раз выполните индексацию из корня модуля:
> ```
> docker compose up -d
> python -m rag_service.ingest --reset
> ```
> Она прочитает `data/manifest.json`, санитизирует документы, порежет на чанки
> и зальёт в Qdrant. В моём прогоне получилось **12 документов → 25 чанков**.


## Антипаттерн 1. Устаревшие данные

Самый частый и самый незаметный. Регламент обновили, старую версию из индекса
**не удалили** — и теперь в базе живут два ответа на один вопрос. Модель
процитирует тот, что найдётся первым. Для сотрудника это выглядит как
«ассистент врёт», хотя врёт не модель, а индекс.

В корпусе «Векторики» это подстроено нарочно: документ об отпуске есть в
**двух редакциях** (см. `data/manifest.json`).


In [ ]:
manifest = json.loads((DATA_DIR / "manifest.json").read_text(encoding="utf-8"))
print("Документы policy-vacation в корпусе:\n")
for doc in manifest["documents"]:
    if doc["doc_id"] == "policy-vacation":
        flag = "✅ действующая" if doc["is_current"] else "⛔ УСТАРЕЛА"
        print(f"  {doc['file']}: версия {doc['version']}, обновлён {doc['updated_at']} — {flag}")
        print(f"     «{doc['title']}»")

Обе редакции лежат в индексе. Разница — принципиальная: в редакции 2023 года
отпуск **28 дней** и заявка **за 14 дней**, в редакции 2026 — **31 день**
(28 + 3 «на выгорание») и заявка **за 21 день**. Посмотрим, что вернёт поиск
без фильтра свежести и с ним.

In [ ]:
from rag_service.embeddings import get_embedder
from rag_service.access import DEMO_PRINCIPALS
from rag_service.retrieval import retrieve, LexicalIndex

if QDRANT_UP:
    emb = get_embedder()
    lex = LexicalIndex.from_kb(kb)
    employee = DEMO_PRINCIPALS["demo-employee-token"]
    q = "Сколько дней отпуска положено в 2026 году?"

    # без фильтра свежести — в выдаче обе версии
    r_all = retrieve(q, employee, kb, emb, lexical=lex, only_current=False)
    print("only_current=False (как у наивного индекса):")
    for h in r_all.hits:
        v = h.metadata.get("version"); cur = h.metadata.get("is_current")
        print(f"  {h.doc_id} v{v} is_current={cur} sim={h.similarity}")

    # с фильтром — только действующая редакция
    r_cur = retrieve(q, employee, kb, emb, lexical=lex, only_current=True)
    print("\nonly_current=True (наш дефолт):")
    for h in r_cur.hits:
        v = h.metadata.get("version")
        cur = h.metadata.get("is_current")
        print(f"  {h.doc_id} v{v} is_current={cur} sim={h.similarity}")
else:
    print("Qdrant не поднят — пропускаем живой поиск.")

В моём прогоне без фильтра в топ-2 попали **обе** редакции с почти
одинаковым скором (0.167) — эмбеддингу всё равно, что одна устарела три года
назад. С фильтром `is_current=True` устаревшая редакция исчезает из выдачи.

Теперь — цена ошибки на живой модели. Спросим qwen то же самое.

In [ ]:
from rag_service.pipeline import RagPipeline

if QDRANT_UP and OLLAMA_UP:
    pipe = RagPipeline(kb=kb, embedder=emb, llm=OllamaLLM())
    pipe.build_lexical_index()
    res = pipe.ask("Сколько дней отпуска мне положено в 2026 году?",
                   DEMO_PRINCIPALS["demo-employee-token"])
    print("Ответ (с фильтром свежести):\n ", res.answer)
    print("\nИсточники:", [(s["n"], s["doc_id"], s["updated_at"]) for s in res.sources])
else:
    print("Нужны Qdrant + Ollama. Ожидаемый ответ с фильтром: «31 календарный день [#1]»,")
    print("источник — редакция 2026. Без фильтра модель легко процитирует 28 дней из 2023.")

В моём прогоне модель ответила **«31 календарный день [#1]»** и сослалась
на редакцию 2026 — потому что устаревший документ до неё просто не дошёл.

**Как лечить.** Устаревание — задача **метаданных**, а не модели:
- каждому документу — `version`, `updated_at`, `is_current`;
- фильтр `is_current=True` по умолчанию (мы так и делаем в `build_filter`);
- при переиндексации — детерминированные id (`doc_id:vN:чанк`), чтобы новая
  версия **перезаписывала**, а не добавлялась к старой;
- правило разрешения конфликтов в промпте: «при расхождении опирайся на более
  свежую дату». Полный разбор версионирования — урок 4.5.


## Антипаттерн 2. Плохой чанкинг

Чанк — это то, что модель увидит целиком или не увидит вовсе. Порежете
неудачно — релевантный факт окажется размазан по двум чанкам, и ни один
не наберёт достаточный скор. Классика из модуля 2: инструкция из 6 шагов,
порезанная посередине, теряет половину шагов.

Посмотрим, как наш ingest порезал документ о VPN.

In [ ]:
if QDRANT_UP:
    from rag_service.store import Hit
    vpn_chunks = [h for h in kb.fetch_all() if h.doc_id == "guide-vpn"]
    vpn_chunks.sort(key=lambda h: h.metadata.get("chunk_index", 0))
    print(f"guide-vpn порезан на {len(vpn_chunks)} чанк(ов):\n")
    for h in vpn_chunks:
        idx = h.metadata.get("chunk_index")
        print(f"— чанк #{idx} ({len(h.text)} символов):")
        print(f"  «{h.short(120)}»\n")
else:
    print("Qdrant не поднят. Правило чанкинга: режем по абзацам (естественная")
    print("смысловая граница), склеиваем до ~700 символов, с перекрытием ~120.")


Наш `chunk_text` (см. `rag_service/ingest.py`) режет **по абзацам**, а не
по фиксированному окну символов: абзац — естественная смысловая единица, как
функция в коде. Длинные абзацы дорезаются по границам предложений, между
чанками оставляется перекрытие, чтобы факт на стыке не потерялся.

**Как диагностировать плохой чанкинг:**
- ищете факт, который **точно есть** в документе, а top-1 его не находит;
- открываете чанки этого документа (как выше) и видите, что факт разорван;
- признак «слишком мелко»: заголовок документа стал отдельным чанком и
  перетягивает скор на себя (мы ловили это в модуле 2 на TF-IDF).

Тонкая настройка чанкинга (контекстные заголовки, semantic chunking,
parent-document) — модуль 2, уроки 2.5–2.6. Здесь важно уметь **увидеть**,
что чанкинг стал причиной промаха.


## Антипаттерн 3. Нерелевантный поиск (семантический разрыв)

Самая коварная поломка: индекс свежий, чанки хорошие, а поиск всё равно
приносит не то. Причина — **разрыв между словами вопроса и словами документа**.

Наш дефолтный эмбеддер — TF-IDF (уроки 2.5/3.2): он сравнивает **слова**,
а не смысл. Спросите официальными терминами регламента — найдёт. Спросите
по-человечески — промахнётся. Проверим на живом примере.

In [ ]:
if QDRANT_UP:
    from rag_service.retrieval import format_hits
    # В документе policy-remote написано «из-за границы до 90 дней».
    # Слова «Грузия» и «два месяца» там нет — TF-IDF их не свяжет.
    q = "Можно ли работать из Грузии два месяца?"
    r = retrieve(q, employee, kb, emb, lexical=lex)
    print("Вопрос:", q)
    print("Нужен документ: policy-remote\n")
    print(format_hits(r.hits[:4]))
    ranks = [h.doc_id for h in r.hits]
    pos = ranks.index("policy-remote") + 1 if "policy-remote" in ranks else None
    print(f"\npolicy-remote на позиции: {pos}")
else:
    print("Qdrant не поднят. В моём прогоне top-1 был guide-onboarding (мимо),")
    print("а policy-remote проваливался вниз с sim≈0.03 — TF-IDF не видит синонимы.")


В моём прогоне top-1 оказался `guide-onboarding` — совершенно не тот
документ, а `policy-remote` провалился на 4-е место с косинусом ~0.03. Для
пользователя это «ассистент не знает про удалёнку», хотя документ в базе есть.

Обратная сторона той же медали — **точные строки**, где, наоборот, силён
лексический поиск, а эмбеддинги плывут. Спросим IP-адрес DNS из VPN-гайда.

In [ ]:
if QDRANT_UP:
    q = "10.10.0.53"
    r = retrieve(q, employee, kb, emb, lexical=lex)
    print("Вопрос (точная строка):", q)
    print(format_hits(r.hits[:3]))
    print("\nЗаметьте колонки sim (плотный поиск) и bm25 (лексический):")
    print("точную строку вытягивает именно BM25, эмбеддинг по цифрам почти бесполезен.")
else:
    print("В моём прогоне '10.10.0.53' нашёлся через BM25 (bm25≈2.9), тогда как")
    print("плотный косинус по цифрам был низким. Точные совпадения — работа BM25.")


Вывод: **ни один способ поиска не покрывает все запросы.** Плотный
(эмбеддинги) понимает смысл, но слеп к точным строкам и жаргону. Лексический
(BM25) ловит термины и коды, но не видит синонимов. Отсюда — гибридный поиск
(dense + BM25 + слияние RRF), реранкинг и multi-query. Это целиком **урок 4.4**;
здесь мы лишь научились **видеть** семантический разрыв в выдаче.


## Антипаттерн 4. Длинный контекст

Соблазн простой: «нашлось много релевантного — засунем всё в промпт, модель
разберётся». Не разберётся. У длинного контекста три цены:

1. **Деньги и время.** Больше токенов на входе — дольше генерация, дороже
   вызов (на API — буквально в деньгах).
2. **«Lost in the middle».** Модели хуже используют факты из **середины**
   длинного контекста, чем из начала и конца — это воспроизводимый эффект.
3. **Разбавление.** Каждый лишний чанк-«почти по теме» — это шум, который
   уводит внимание модели от нужного факта.

Посмотрим на рост числа входных токенов при увеличении числа чанков.

In [ ]:
from rag_service.prompts import build_prompt

if QDRANT_UP:
    q = "Как оформить отпуск и за сколько дней подавать заявку?"
    r = retrieve(q, employee, kb, emb, lexical=lex, final_k=8)
    print(f"{'чанков':>7} | {'символов в промпте':>18} | примерно токенов")
    print("-" * 48)
    for k in (1, 2, 4, 6, 8):
        system, user, mapping, ctx = build_prompt(q, r.hits[:k], hardened=True, canary="X")
        total_chars = len(system) + len(user)
        print(f"{len(mapping):>7} | {total_chars:>18} | ~{total_chars // 4}")
    print("\n(грубая оценка: ~4 символа на токен для кириллицы через tiktoken-подобные BPE)")
else:
    print("Qdrant не поднят. Идея: каждый добавленный чанк линейно растит промпт.")
    print("В моём прогоне /ask с 4 чанками давал ~991 prompt_tokens, с большим числом")
    print("документов (запрос HR) — уже ~1518. Контекст надо СУЖАТЬ, а не расширять.")


**Как лечить длинный контекст:**
- держать `final_k` маленьким (у нас дефолт 4) и **реранкать** кандидатов,
  чтобы в эти 4 места попали лучшие — а не первые попавшиеся (урок 4.4);
- ставить `max_context_chars` как жёсткий бюджет (у нас 4000) — `build_context`
  обрежет лишнее, а не отправит модели «простыню»;
- класть самый релевантный чанк **первым** (после реранкинга он там и будет);
- если ответ требует много источников — это сигнал пересмотреть чанкинг,
  а не раздувать контекст.

Меньше, но точнее — почти всегда лучше, чем больше, но мутнее.


## Антипаттерн 5. Тихие сбои без наблюдаемости

Он мета по отношению к первым четырём. Устаревший документ, семантический
промах, раздутый контекст — **ни один не бросает исключение**. Сервис жив,
`/healthz` зелёный. Поломку замечают, когда приходит злой тикет от сотрудника —
то есть через дни или недели.

Единственная защита — **наблюдаемость**: видеть не «упал / не упал», а
*распределение* качества и латентности во времени. Наш сервис уже пишет
структурные логи и считает метрики — загляните в стадии одного запроса.

In [ ]:
from rag_service.observability import configure_logging

if QDRANT_UP and OLLAMA_UP:
    configure_logging("INFO", json_logs=False, service="rag")  # видно стадии в ноутбуке
    res = pipe.ask("За сколько дней подавать заявку на отпуск?",
                   DEMO_PRINCIPALS["demo-employee-token"])
    print("\nИтог:", res.outcome, "| источников:", len(res.sources))
    print("Каждая строка stage.done выше — отдельная стадия с таймингом.")
    print("Это и есть трейс запроса. Prometheus-метрики и алерты — урок 4.3.")
else:
    print("Нужны Qdrant + Ollama. Идея: каждый запрос оставляет трейс из стадий")
    print("(retrieve.dense, retrieve.bm25, llm.generate, guard.answer) с длительностями.")
    print("В моём прогоне: dense ~16-20мс, bm25 ~0.1мс, llm ~1.7-2.5с. Разбор — урок 4.3.")

**Сводная таблица антипаттернов** — держите её под рукой как чек-лист
для code review вашего RAG:

| # | Антипаттерн | Симптом | Как увидеть | Где лечим |
|---|---|---|---|---|
| 1 | Устаревшие данные | модель цитирует старьё | две версии в выдаче | 4.5 (метаданные) |
| 2 | Плохой чанкинг | факт «есть, но не находится» | открыть чанки документа | 2.5–2.6 |
| 3 | Нерелевантный поиск | top-1 не по теме | `format_hits`, семант. разрыв | 4.4 (гибрид+rerank) |
| 4 | Длинный контекст | медленно, «размыто» | рост prompt_tokens | 4.4 (rerank), бюджет |
| 5 | Тихие сбои | тикет вместо алерта | трейс + метрики | 4.3 (мониторинг) |


## ⚠️ Частые ошибки

- **«Обновлю документ — RAG сам подхватит».** Не подхватит, если старую версию
  не удалить и не проставить `is_current`. Индекс — не файловая система с
  перезаписью; без детерминированных id старое и новое лежат рядом.
- **«Добавлю ещё чанков в промпт — точность вырастет».** Обычно наоборот:
  растёт латентность и шум, а нужный факт тонет в середине.
- **«Демо работает — значит, готово».** Демо — это 5 удобных вопросов на чистых
  данных. Прод — это кривые формулировки, жаргон, устаревшие документы и
  отравленный UGC (следующий урок).
- **«Порог релевантности спасёт от мусора».** Порог не переносится между
  эмбеддерами и не ловит «уверенный мусор» (документ с высоким скором, но не по
  теме). Мы это подробно разбирали в уроке 3.4.
- **`localhost` вместо `127.0.0.1`.** На Windows httpx сначала пробует IPv6 `::1`
  и теряет ~2 секунды на каждом свежем соединении к Ollama/Qdrant (грабля 3.2).


## Упражнения

1. **Детектор дублей версий.** Напишите функцию, которая по `kb.fetch_all()`
   находит все `doc_id`, у которых в индексе больше одной версии, и выводит,
   какая помечена `is_current`. Это готовый health-check «в индексе нет
   устаревших дублей».

2. **Карта семантических разрывов.** Возьмите 3–4 вопроса «по-человечески»
   («отгул на свадьбу», «комп сломался», «работаю из Тбилиси») и для каждого
   выведите top-3 через `format_hits`. Отметьте, где top-1 промахнулся, и
   предположите, поможет ли BM25 (точные слова) или нужен смысл (эмбеддинг e5).

3. **Бюджет контекста.** Постройте таблицу: как меняется число prompt-токенов
   и время генерации при `final_k` = 1, 2, 4, 8 на одном и том же вопросе
   (нужна живая Ollama). Найдите точку, где ответ уже не улучшается, а токены
   всё растут.


## Мини-квиз

<details><summary><b>1. Почему устаревший документ опаснее, чем отсутствующий?</b></summary>

Отсутствие документа даёт честный отказ «нет ответа». Устаревший документ даёт
**уверенный неправильный ответ** с цитатой — он выглядит достоверно, и ошибку
никто не проверяет. Плюс он конкурирует за место в выдаче с актуальной версией.
</details>

<details><summary><b>2. TF-IDF нашёл «10.10.0.53», но промахнулся на «работе из Грузии». Почему?</b></summary>

TF-IDF сравнивает **слова**. «10.10.0.53» дословно есть в документе — лексическое
совпадение идеально. А «Грузия» в документе нет вовсе (там «из-за границы до 90
дней») — общих слов нет, скор около нуля. Смысловую связь видит только
семантический эмбеддер (e5) или переформулировка запроса (multi-query, урок 4.4).
</details>

<details><summary><b>3. Все пять антипаттернов объединяет одно свойство. Какое?</b></summary>

Они **не бросают исключений**. Сервис не падает, `/healthz` зелёный, латентность
в норме — деградирует только *качество ответов*. Поэтому антипаттерн №5
(отсутствие наблюдаемости) — самый опасный: без метрик остальные четыре
невидимы до первого злого тикета.
</details>

<details><summary><b>4. Почему «добавить больше чанков в контекст» — плохая идея по умолчанию?</b></summary>

Три причины: (1) линейный рост токенов = медленнее и дороже; (2) «lost in the
middle» — модель хуже использует факты из середины длинного контекста;
(3) каждый лишний «почти релевантный» чанк разбавляет внимание модели шумом.
Лучше меньше чанков, но точно отобранных реранкингом.
</details>

<details><summary><b>5. Документ обновили в HR-портале. Что должно произойти в индексе, чтобы RAG не начал врать?</b></summary>

Новая версия должна **перезаписать** старую по детерминированному id (или старая
— получить `is_current=False`), корпус — переиндексироваться, а фильтр поиска —
по умолчанию отбирать только `is_current=True`. Иначе обе версии останутся в
выдаче и модель процитирует случайную.
</details>


## Решения

Сначала попробуйте сами — 20 минут честной попытки стоят больше, чем чтение
готового. Потом сверяйтесь.

**Решение 1 — детектор дублей версий.** Обходим `kb.fetch_all()` один раз и
группируем чанки по `(doc_id, version)`. Подозрительны два случая: у документа
больше одной версии в индексе (устаревшую забыли удалить — антипаттерн 1
в чистом виде) и — хуже — число версий с `is_current=True` не равно единице
(сломан сам манифест: фильтр свежести либо вернёт обе редакции, либо ни одной).
Такой функции самое место в CI или в `/health`-проверке рядом с числом чанков:

In [ ]:
from collections import defaultdict


def find_version_duplicates(kb) -> dict:
    """doc_id -> {version: {chunks, is_current}} для doc_id с >1 версией в индексе."""
    versions: dict = defaultdict(dict)
    for hit in kb.fetch_all():
        v = int(hit.metadata.get("version", 1))
        row = versions[hit.doc_id].setdefault(
            v, {"chunks": 0, "is_current": bool(hit.metadata.get("is_current", True))})
        row["chunks"] += 1
    return {doc_id: vers for doc_id, vers in versions.items() if len(vers) > 1}


if QDRANT_UP:
    dups = find_version_duplicates(kb)
    if not dups:
        print("OK: дублей версий в индексе нет")
    for doc_id, vers in dups.items():
        print(f"{doc_id}: версий в индексе - {len(vers)}")
        for v, row in sorted(vers.items()):
            mark = "текущая" if row["is_current"] else "УСТАРЕЛА, но лежит в индексе"
            print(f"  v{v}: {row['chunks']} чанков - {mark}")
        currents = [v for v, row in vers.items() if row["is_current"]]
        if len(currents) != 1:      # 0 или 2+ текущих - сломан сам манифест
            print(f"  !!! is_current=True у {len(currents)} версий - чинить немедленно")
else:
    print("Qdrant не поднят. В моём прогоне: policy-vacation - 2 версии,")
    print("v1 (2 чанка) устарела, но лежит в индексе; v2 - текущая.")

Детектор честно поймал наш заготовленный дубль: обе редакции
`policy-vacation` в индексе, старая помечена `is_current=False`. Фильтр
свежести нас страхует, но правильное лечение из антипаттерна 1 — удалить
старую версию при переиндексации; health-check как раз напомнит об этом.

**Решение 2 — карта семантических разрывов.** Гоняем «человеческие» вопросы
через полный `retrieve` (плотный TF-IDF + BM25 + RRF) и смотрим, куда попал
нужный документ. Четвёртый вопрос — контрольный: слова в нём совпадают
с текстом регламента, лексика обязана справиться:

In [ ]:
from rag_service.retrieval import format_hits

if QDRANT_UP:
    cases = [
        ("Хочу взять отгул на свадьбу", "policy-vacation"),
        ("У меня комп сломался, что делать?", "policy-equipment"),
        ("Могу я работать из Тбилиси?", "policy-remote"),
        ("За сколько дней подавать заявку на отпуск?", "policy-vacation"),  # контроль
    ]
    for q, expected in cases:
        r = retrieve(q, employee, kb, emb, lexical=lex)
        top1 = r.hits[0].doc_id if r.hits else "-"
        verdict = "ok  " if top1 == expected else "МИМО"
        print(f"[{verdict}] {q}   (нужен: {expected})")
        print(format_hits(r.hits[:3]))
        print()
else:
    print("Qdrant не поднят. В моём прогоне: «комп сломался» - top-1 guide-vpn")
    print("(нужный policy-equipment третий), «Тбилиси» - top-1 policy-sickleave")
    print("(policy-remote вообще не в топ-3), контрольный вопрос - уверенный ok.")

Разбор карты (ваши скоры могут отличаться на сотые, картина — нет,
поиск здесь детерминированный):

- **«Отгул на свадьбу»** — формально top-1 верный, но посмотрите на скоры:
  sim≈0.05, bm25≈0.4 — это уровень шума, совпали служебные слова, а не смысл.
  Слов «отгул» и «свадьба» в корпусе **нет вообще** — тут не разрыв поиска,
  а дыра в базе знаний. Не спасёт ни BM25, ни e5: ответа не существует.
  Правильное поведение системы — честный отказ (порог!), а сигнал для
  владельца базы — пополнить регламент.
- **«Комп сломался»** — промах: в документе «при полОМКе оборудования»,
  в вопросе «слОМАЛся». TF-IDF и BM25 работают точными токенами — однокоренные
  слова для них чужие. Это работа для смысла: e5 свяжет «комп сломался»
  с «поломкой оборудования» (проверим в уроке 4.4).
- **«Тбилиси»** — промах: города в корпусе нет, в `policy-remote` написано
  «из-за границы до 90 дней». BM25 бессилен по определению; нужен эмбеддинг,
  который знает, что Тбилиси — это заграница. Такое умеет e5 (и то не
  гарантированно — маленькая модель), а надёжнее всего — reranker из 4.4.
- **Контрольный вопрос** — уверенное попадание (sim≈0.36, bm25≈8.7): когда
  слова совпадают, лексика работает отлично и дёшево.

Мораль: такая карта из пяти строк кода — самый дешёвый способ решить,
**нужен ли вам семантический эмбеддер**, до того как платить за него
скачиванием, памятью и миллисекундами (урок 4.4).

**Решение 3 — бюджет контекста.** Один и тот же вопрос, один и тот же список
кандидатов, меняем только число чанков в промпте. Считаем всё честно, из
телеметрии Ollama: `prompt_eval_count`, `eval_count` и длительность
(`max_chars` поднят, чтобы контекст не обрезался и рост был честным):

In [ ]:
if QDRANT_UP and OLLAMA_UP:
    llm = OllamaLLM()
    q = "Как оформить отпуск и за сколько дней подавать заявку?"
    r = retrieve(q, employee, kb, emb, lexical=lex, final_k=8)
    rows = []
    for k in (1, 2, 4, 8):
        system, user, mapping, ctx = build_prompt(q, r.hits[:k], hardened=True,
                                                  max_chars=8000)
        res = llm.chat(system, user, max_tokens=200)
        rows.append((len(mapping), res))

    print(f"{'чанков':>6} | {'prompt_tok':>10} | {'ответ_tok':>9} | {'время, с':>8}")
    print("-" * 44)
    for k, res in rows:
        print(f"{k:>6} | {res.prompt_tokens:>10} | {res.completion_tokens:>9} "
              f"| {res.duration_s:>8.1f}")

    print("\nОтветы (первые 220 символов):")
    for k, res in rows:
        print(f"\n--- k={k} ---")
        print(res.text.replace("\n", " ")[:220])
else:
    print("Нужны Qdrant + Ollama. В моём прогоне prompt-токены выросли с ~750")
    print("(k=1) до ~2350 (k=8) - в три раза, а ответ перестал улучшаться уже")
    print("на k=1: нужный чанк и так стоял первым.")

Разбор бюджета (числа — из прогона выше; ваши будут другими, тренд — нет):

- **Промпт-токены растут линейно**: ~750 при одном чанке → ~2357 при восьми,
  втрое. На платном API это прямые деньги (калькулятор из урока 1.3), на
  локальной модели — место в контекстном окне: восемь чанков занимают больше
  половины `num_ctx=4096`, и это ещё без истории диалога.
- **Точка насыщения — k=1.** Верхний чанк `policy-vacation` уже содержит и
  «31 день», и «заявку за 21 день» — ответ полный по обеим половинам вопроса.
  Всё, что дальше, — расход без пользы. А иногда и во вред: в соседнем прогоне
  на k=4 модель пересказала «3 дополнительных дня „на выгорание“» как «дни
  доверия» — термин, которого нет ни в одном документе. Чем больше в контексте
  постороннего, тем небрежнее формулировки у 3B-модели.
- **Время генерации от промпта почти не зависит** (4.7с → 3.8с, на k=4 и вовсе
  2.9с — короткий ответ): у Ollama оно определяется числом токенов ОТВЕТА,
  prefill на GPU дёшев. Не дайте этому себя обмануть: платите вы всё равно —
  деньгами на API, местом в окне и риском «lost in the middle».

Почему тогда дефолт сервиса `final_k=4`, а не 1? Страховка на вопросы, где
ретривер промахнулся рангом: в карте разрывов выше нужный `policy-equipment`
стоял третьим — с `final_k=1` модель его не увидела бы вовсе. Четыре чанка —
компромисс «сколько полезно», а правильное направление оптимизации — не
расширять контекст, а точнее ранжировать: reranker в уроке 4.4 сжимает
20 кандидатов в 4 точных.

## Итоги

- Продовый RAG ломается **тихо**: не исключениями, а деградацией качества.
- Пять типовых антипаттернов и как их **увидеть** на живой базе:
  устаревшие данные → две версии в выдаче; плохой чанкинг → факт разорван;
  нерелевантный поиск → семантический разрыв в `format_hits`; длинный контекст
  → рост prompt-токенов; тихие сбои → отсутствие трейсов и метрик.
- Каждая болезнь лечится конкретным уроком модуля 4 — держите карту-таблицу
  как чек-лист.
- Мы **никого не чинили** в этом уроке — только ставили диагноз. Лечение
  начинается со следующего урока: **защита RAG от индексации до ответа**,
  где нас ждёт живая атака prompt injection через отравленный документ.

## Что дальше

Урок **4.2 «Защита RAG: от индексации до ответа»** — prompt injection через
документы, санитизация, контроль доступа и PII. Там мы увидим, как наш же
сервис на наивном промпте **выдаёт выдуманный пароль**, и как эшелонированная
оборона это останавливает.

## Полезные ссылки
- Lost in the Middle (Liu et al., 2023): <https://arxiv.org/abs/2307.03172>
- Qdrant — фильтрация по метаданным: <https://qdrant.tech/documentation/concepts/filtering/>
- Модуль 2 (чанкинг, гибридный поиск) и модуль 3 (пороги, оценка) — база под этот урок.
